This notebook contains Python and R code for reproducing the results in our paper on using a large language model as a filter to identify questions that students perceive as higher quality for automatic fill-in-the-blank cloze question generation:

***UPDATE***Dittel, J. S., Van Campenhout, R., & Johnson, B. G. (2025). Refining sentence selection for automatic cloze question generation with large language models. In _Proceedings of the Twelfth ACM Conference on Learning at Scale (L@S '25)_, Palermo, Italy. https://doi.org/10.1145/3698205.3733926, **pp. TODO: add page numbers when available**

Results are presented in the order they occur, organized by the paper's sections. For each result, an excerpt from the paper is given followed by code to compute the result from the data set provided. Example:

>While the original dataset contained over 5.2 million student-question sessions, the present analysis focuses on textbooks from three publishers who granted permission for generative AI research. Filtering data to these textbooks yields 1,305,957 sessions across 210,902 questions, 106,183 students, and 2,510 textbooks.

```len( sessions ), sessions.question_id.nunique(), sessions.student_id.nunique(), sessions.textbook_id.nunique()```

Please refer to the paper for additional context.

In [1]:
import pandas as pd

In [2]:
%load_ext rpy2.ipython

## Read dataset

In [3]:
sessions = pd.read_parquet( 'sessions.parquet' )
sessions.head()

,student_id,question_id,textbook_id,subject,thumbs_up,thumbs_down,H1_first_correct,H2_cumulative_answered,H3_spelling_suggestion,H4_sentence_textrank_rank,H5_answer_tf_idf_rank,H6_answer_pos,H7_answer_log_probability,H8_answer_location,H9_feedback,H10_reviewed,H11_llm_aligned
0,26EFUDCGXGK2R2BMUA65,000005fcee0aca01fd16f0fcaebbd46bbcef1131bdedab...,9781071875674,Political Science,0,0,0,1,0,0.226576,0.011482,NOUN,-13.244523,9,outcome,0,0
1,AK53WK5B75TBJXS7MMSM,000005fcee0aca01fd16f0fcaebbd46bbcef1131bdedab...,9781071875674,Political Science,0,0,0,1,0,0.226576,0.011482,NOUN,-13.244523,9,outcome,0,0
2,VGEWC8NNTPF6ZNP2XGDM,000005fcee0aca01fd16f0fcaebbd46bbcef1131bdedab...,9781071875674,Political Science,0,0,0,1,0,0.226576,0.011482,NOUN,-13.244523,9,outcome,0,0
3,3G5CA6NTMHFZXTK4EQZK,00001bc7b94e02d63ccd61bafe8082b473c27ffb7c5d40...,9781506318134,Political Science,0,0,1,1,0,0.231308,0.014590,NOUN,-11.116641,6,common_answer,0,0
4,5XNZ4CZRKQREESC27MWR,00001bc7b94e02d63ccd61bafe8082b473c27ffb7c5d40...,9781506318134,Political Science,0,0,1,1,0,0.231308,0.014590,NOUN,-11.116641,6,common_answer,0,0


In [4]:
questions = pd.read_parquet( 'questions.parquet' )
questions.head()

,question_id,textbook_id,subject,students,thumbs_up,thumbs_down,stem,answer,sentence,llm_stem,llm_answer,H11_llm_aligned
0,0000abde625a9e1e9cec5dbdae42055a3642b65efb6d09...,9781544356433,Social Science,75,0,0,"Debra worked at a McDonald's and, lacking othe...",childcare,"Debra worked at a McDonald's and, lacking othe...","Debra worked at a McDonald's and, lacking othe...",childcare,1
1,00010f79d65317116ff1a972de55711223065dc1bf6571...,9781317217381,Psychology,1,0,0,The general failure of the ______ theories to ...,universalist,The general failure of the universalist theori...,The general failure of the ______ theories to ...,universalist,1
2,000380395e0b1f9a2c9cadb955e0ebd6ca79142f59f4c6...,9781351754705,History,5,0,0,After Justinian officially ended the teaching ...,Athens,After Justinian officially ended the teaching ...,After Justinian officially ended the teaching ...,working,0
3,0007eb409efb5a2cd71ddc2a71310aa2f95651c09ffbae...,9781000800418,Business & Economics,1,0,0,"Regarding social ______, some people believe t...",sustainability,"Regarding social sustainability, some people b...","Regarding social sustainability, some people b...",economic,0
4,000b57422bac7cfdb2d729827f9ac3202d1c33bc124ce1...,9781351754705,History,1,0,0,Many voters may have been mindful of Rome's ne...,southern,Many voters may have been mindful of Rome's ne...,Many voters may have been mindful of Rome's ne...,protect,0


## 2. METHODS

### 2.3 Causal Modeling of Student Ratings

>While the original dataset contained over 5.2 million student-question sessions, the present analysis focuses on textbooks from three publishers who granted permission for generative AI research. Filtering data to these textbooks yields 1,305,957 sessions across 210,902 questions, 106,183 students, and 2,510 textbooks.

In [5]:
len( sessions ), sessions.question_id.nunique(), sessions.student_id.nunique(), sessions.textbook_id.nunique()

(1305957, 210902, 106183, 2510)

>Using the standard BISAC major subject heading classification [30] available for most of the textbooks, the top subject domains as a fraction of session data were Social Science (27.2%), Psychology (22.8%), and Political Science (14.2%).

In [6]:
sessions.subject.value_counts( normalize=True, dropna=False ).apply( lambda p: f'{p:.1%}' )

subject
Social Science                       27.2%
Psychology                           22.8%
Political Science                    14.2%
Business & Economics                 11.5%
Language Arts & Disciplines          11.2%
Education                             6.4%
Law                                   1.4%
Family & Relationships                1.1%
History                               0.6%
Medical                               0.6%
Science                               0.5%
Music                                 0.4%
Sports & Recreation                   0.3%
Nature                                0.3%
Technology & Engineering              0.3%
Performing Arts                       0.2%
Photography                           0.2%
Computers                             0.1%
Philosophy                            0.1%
None                                  0.1%
Health & Fitness                      0.1%
Art                                   0.1%
Architecture                          0.1%
Rel

## 3. RESULTS AND DISCUSSION

### 3.1 LLM-Based Answer Selection

>Of the 210,902 unique questions in the dataset, the LLM-selected blank matched the rule-based system’s blank in 39,748 cases (18.8%). These aligned questions account for 249,450 sessions, which is approximately 19.1% of the entire session dataset.

In [7]:
print( f'{questions.H11_llm_aligned.sum()} {questions.H11_llm_aligned.mean():.1%}' )
print( f'{sessions.H11_llm_aligned.sum()} {sessions.H11_llm_aligned.mean():.1%}' )

39748 18.8%
249450 19.1%


>An illustrative example of how an LLM can incorporate sentence-level nuances into answer selection comes from the following sentence in a chemistry textbook in the dataset [36].
>
>_The electrons in the H–Cl bond of a hydrogen chloride [AQG] molecule spend more time near the chlorine [LLM] atom than near the hydrogen atom._

In [8]:
idx = 206701
row = questions.loc[ idx ]
print( f'Sentence:   {row.sentence}' )
print()
print( f'Rule-based: {row.stem}' )
print( f'Answer:     {row.answer}' )
print()
print( f'LLM-based:  {row.llm_stem}' )
print( f'Answer:     {row.llm_answer}' )
row.to_frame()

Sentence:   The electrons in the H–Cl bond of a hydrogen chloride molecule spend more time near the chlorine atom than near the hydrogen atom.

Rule-based: The electrons in the H–Cl bond of a hydrogen ______ molecule spend more time near the chlorine atom than near the hydrogen atom.
Answer:     chloride

LLM-based:  The electrons in the H–Cl bond of a hydrogen chloride molecule spend more time near the ______ atom than near the hydrogen atom.
Answer:     chlorine


,206701
question_id,f6681e7b46eb58255806d5678d142d2ddd4953dfd12a72...
textbook_id,9781951693817
subject,Science
students,110
thumbs_up,4
thumbs_down,0
stem,The electrons in the H–Cl bond of a hydrogen _...
answer,chloride
sentence,The electrons in the H–Cl bond of a hydrogen c...
llm_stem,The electrons in the H–Cl bond of a hydrogen c...


>Ratings were given in a total of 5,886 sessions, 3,479 thumbs up and 2,407 thumbs down, a rate of 2.66 thumbs up and 1.84 thumbs down per 1,000 sessions (slightly lower than in the original, larger dataset).

In [9]:
rated = ( sessions.thumbs_up | sessions.thumbs_down ).astype( bool )
print( f'Ratings:     {rated.sum()}' )
print( f'Thumbs up:   {sessions.thumbs_up.sum()}' )
print( f'Thumbs down: {sessions.thumbs_down.sum()}' )

Ratings:     5886
Thumbs up:   3479
Thumbs down: 2407


>Of the 106,183 students, 3,289 (3.10%) used the rating feature, and of the 210,902 questions, 4,803 (2.28%) were rated (either thumbs up or thumbs down).

In [10]:
rating_students = sessions[ rated ].student_id.nunique()
rated_questions = sessions[ rated ].question_id.nunique()
print( f'{rating_students} {rating_students / sessions.student_id.nunique():.2%}' )
print( f'{rated_questions} {rated_questions / sessions.question_id.nunique():.2%}' )

3289 3.10%
4803 2.28%


>**Table 2. Ratings per 1,000 sessions based on LLM alignment.**

In [11]:
def rating_rate( rated ):
    return rated.mean().round( 5 ) * 1000

In [12]:
sessions.groupby( 'H11_llm_aligned' ).agg( sessions=( 'student_id', 'count' ),
                                           thumbs_up=( 'thumbs_up', rating_rate ),
                                           thumbs_down=( 'thumbs_down', rating_rate ) )

,sessions,thumbs_up,thumbs_down
H11_llm_aligned,,,
0,1056507,2.58,1.94
1,249450,3.02,1.44


>**Table 3. Mean or proportion of each explanatory variable for aligned and non-aligned question sessions.**

In [13]:
sessions.groupby( 'H11_llm_aligned' ).agg(
    H1_first_correct=( 'H1_first_correct', 'mean' ),
    H3_spelling_suggestion=( 'H3_spelling_suggestion', 'mean' ),
    H4_sentence_textrank_rank=( 'H4_sentence_textrank_rank', 'mean' ),
    H5_answer_tf_idf_rank=( 'H5_answer_tf_idf_rank', 'mean' ),
    H10_reviewed=( 'H10_reviewed', 'mean' ),
).round( 3 )[ ::-1 ].T.set_axis( [ 'aligned', 'non_aligned' ], axis=1 )

,aligned,non_aligned
H1_first_correct,0.591,0.595
H3_spelling_suggestion,0.056,0.049
H4_sentence_textrank_rank,0.275,0.280
H5_answer_tf_idf_rank,0.119,0.140
H10_reviewed,0.027,0.030


In [14]:
sessions.groupby( 'H11_llm_aligned' ).agg(
    H2_cumulative_answered=( 'H2_cumulative_answered', 'mean' ),
    H7_answer_log_probability=( 'H7_answer_log_probability', 'mean' ),
    H8_answer_location=( 'H8_answer_location', 'mean' ),
).round( 1 )[ ::-1 ].T.set_axis( [ 'aligned', 'non_aligned' ], axis=1 )

,aligned,non_aligned
H2_cumulative_answered,44.9,45.6
H7_answer_log_probability,-12.8,-12.3
H8_answer_location,9.7,11.4


In [15]:
sessions.groupby( 'H11_llm_aligned' ).H6_answer_pos.value_counts( normalize=True ).sort_index( ascending=[ False, True ] ).round( 3 )

H11_llm_aligned  H6_answer_pos
1                ADJ              0.323
                 ADV              0.003
                 NOUN             0.633
                 PROPN            0.019
                 VERB             0.021
0                ADJ              0.322
                 ADV              0.007
                 NOUN             0.589
                 PROPN            0.056
                 VERB             0.025
Name: proportion, dtype: float64

In [16]:
sessions.groupby( 'H11_llm_aligned' ).H9_feedback.value_counts( normalize=True ).sort_index( ascending=[ False, True ] ).round( 3 )

H11_llm_aligned  H9_feedback  
1                common_answer    0.621
                 context          0.172
                 outcome          0.208
0                common_answer    0.592
                 context          0.174
                 outcome          0.234
Name: proportion, dtype: float64

>Mixed effects logistic regression was performed with random intercepts for students to test the new hypothesis H11 while accounting for known causal factors. As in previous studies, the student intercept choice was supported by a significantly lower BIC (23,212.5 vs. 28,420.0), with coefficients consistent with a question intercept model in direction and magnitude.

In [17]:
%%R
library( arrow )
library( glmmTMB )

Some features are not enabled in this build of Arrow. Run `arrow_info()` for more information.
The repository you retrieved Arrow from did not include all of Arrow's features.
You can install a fully-featured version by running:
`install.packages('arrow', repos = 'https://apache.r-universe.dev')`.

Attaching package: ‘arrow’

The following object is masked from ‘package:utils’:

    timestamp



In [18]:
%%R
sessions <- read_parquet( 'sessions.parquet' )
str( sessions )

Classes ‘tbl_df’, ‘tbl’ and 'data.frame':	1305957 obs. of  17 variables:
 $ student_id               : chr  "26EFUDCGXGK2R2BMUA65" "AK53WK5B75TBJXS7MMSM" "VGEWC8NNTPF6ZNP2XGDM" "3G5CA6NTMHFZXTK4EQZK" ...
 $ question_id              : chr  "000005fcee0aca01fd16f0fcaebbd46bbcef1131bdedab888522e6de2ced78c8" "000005fcee0aca01fd16f0fcaebbd46bbcef1131bdedab888522e6de2ced78c8" "000005fcee0aca01fd16f0fcaebbd46bbcef1131bdedab888522e6de2ced78c8" "00001bc7b94e02d63ccd61bafe8082b473c27ffb7c5d4073211fa78cc9318134" ...
 $ textbook_id              : chr  "9781071875674" "9781071875674" "9781071875674" "9781506318134" ...
 $ subject                  : chr  "Political Science" "Political Science" "Political Science" "Political Science" ...
 $ thumbs_up                : int  0 0 0 0 0 0 0 0 0 0 ...
 $ thumbs_down              : int  0 0 0 0 0 0 0 0 0 0 ...
 $ H1_first_correct         : int  0 0 0 1 1 0 1 1 0 1 ...
 $ H2_cumulative_answered   : int  1 1 1 1 1 1 1 1 1 1 ...
 $ H3_spelling_suggestion   : i

Thumbs down question intercepts model for comparison of BIC (28,420.0) and coefficients with student intercepts model used in this work (Table 4).

In [19]:
%%R
model <- glmmTMB( thumbs_down ~ H1_first_correct
                              + H2_cumulative_answered
                              + H3_spelling_suggestion
                              + H4_sentence_textrank_rank
                              + H5_answer_tf_idf_rank
                              + H6_answer_pos
                              + H7_answer_log_probability
                              + H8_answer_location
                              + H9_feedback
                              + H10_reviewed
                              + H11_llm_aligned
                              + (1|question_id),
                              family=binomial(link=logit), data=sessions )
summary( model )

 Family: binomial  ( logit )
Formula:          
thumbs_down ~ H1_first_correct + H2_cumulative_answered + H3_spelling_suggestion +  
    H4_sentence_textrank_rank + H5_answer_tf_idf_rank + H6_answer_pos +  
    H7_answer_log_probability + H8_answer_location + H9_feedback +  
    H10_reviewed + H11_llm_aligned + (1 | question_id)
Data: sessions

      AIC       BIC    logLik  deviance  df.resid 
  28214.6   28420.0  -14090.3   28180.6   1305940 

Random effects:

Conditional model:
 Groups      Name        Variance Std.Dev.
 question_id (Intercept) 69.06    8.31    
Number of obs: 1305957, groups:  question_id, 210902

Conditional model:
                            Estimate Std. Error z value Pr(>|z|)    
(Intercept)               -1.106e+01  4.548e-01 -24.326  < 2e-16 ***
H1_first_correct          -8.313e-01  5.421e-02 -15.335  < 2e-16 ***
H2_cumulative_answered    -5.515e-03  7.233e-04  -7.624 2.45e-14 ***
H3_spelling_suggestion    -6.607e-01  1.486e-01  -4.445 8.79e-06 ***
H4_sentenc

>**Table 4. Thumbs down regression model.**

Thumbs down students intercepts model including H1–H11.

In [20]:
%%R
model <- glmmTMB( thumbs_down ~ H1_first_correct
                              + H2_cumulative_answered
                              + H3_spelling_suggestion
                              + H4_sentence_textrank_rank
                              + H5_answer_tf_idf_rank
                              + H6_answer_pos
                              + H7_answer_log_probability
                              + H8_answer_location
                              + H9_feedback
                              + H10_reviewed
                              + H11_llm_aligned
                              + (1|student_id),
                              family=binomial(link=logit), data=sessions )
summary( model )

 Family: binomial  ( logit )
Formula:          
thumbs_down ~ H1_first_correct + H2_cumulative_answered + H3_spelling_suggestion +  
    H4_sentence_textrank_rank + H5_answer_tf_idf_rank + H6_answer_pos +  
    H7_answer_log_probability + H8_answer_location + H9_feedback +  
    H10_reviewed + H11_llm_aligned + (1 | student_id)
Data: sessions

      AIC       BIC    logLik  deviance  df.resid 
  23007.0   23212.5  -11486.5   22973.0   1305940 

Random effects:

Conditional model:
 Groups     Name        Variance Std.Dev.
 student_id (Intercept) 100.2    10.01   
Number of obs: 1305957, groups:  student_id, 106183

Conditional model:
                            Estimate Std. Error z value Pr(>|z|)    
(Intercept)               -1.164e+01  2.294e-01  -50.74  < 2e-16 ***
H1_first_correct          -1.195e+00  5.737e-02  -20.83  < 2e-16 ***
H2_cumulative_answered    -2.369e-03  6.961e-04   -3.40 0.000665 ***
H3_spelling_suggestion    -5.165e-01  1.507e-01   -3.43 0.000608 ***
H4_sentence_te

Odds ratios.

In [21]:
%%R
exp( fixef( model )$cond )

              (Intercept)          H1_first_correct    H2_cumulative_answered 
             8.812870e-06              3.027427e-01              9.976336e-01 
   H3_spelling_suggestion H4_sentence_textrank_rank     H5_answer_tf_idf_rank 
             5.965864e-01              3.510819e+00              2.170948e+00 
         H6_answer_posADV         H6_answer_posNOUN        H6_answer_posPROPN 
             3.795370e+00              1.276073e+00              1.981106e+00 
        H6_answer_posVERB H7_answer_log_probability        H8_answer_location 
             1.959316e+00              1.093730e+00              9.893650e-01 
       H9_feedbackcontext        H9_feedbackoutcome              H10_reviewed 
             1.035461e+00              1.229748e+00              9.350284e-01 
          H11_llm_aligned 
             6.872091e-01 


>Adding H11_llm_aligned to a model already including H1–H10 decreases BIC from 23,227.0 to 23,212.5 (14.5 points), confirming improved explanatory power.

Thumbs down students intercepts model including only H1–H10 to verify BIC = 23,227.0. BIC for full model (23,212.5) is seen above.

In [22]:
%%R
model <- glmmTMB( thumbs_down ~ H1_first_correct
                              + H2_cumulative_answered
                              + H3_spelling_suggestion
                              + H4_sentence_textrank_rank
                              + H5_answer_tf_idf_rank
                              + H6_answer_pos
                              + H7_answer_log_probability
                              + H8_answer_location
                              + H9_feedback
                              + H10_reviewed
                              + (1|student_id),
                              family=binomial(link=logit), data=sessions )
BIC( model )

[1] 23227.04


>In a coefficient stability check, a pared-down model containing only H11_llm_aligned and student intercepts yields an odds ratio of 0.636; introducing all ten controls shifts it to 0.687 (an 8% change), indicating the effect is stable rather than artifactual.

Pared-down model to verify H11_llm_aligned OR = 0.636. H11_llm_aligned OR = 0.687 in full model is seen above.

In [23]:
%%R
model <- glmmTMB( thumbs_down ~ H11_llm_aligned
                              + (1|student_id),
                              family=binomial(link=logit), data=sessions )
exp( fixef( model )$cond )

    (Intercept) H11_llm_aligned 
   3.135723e-06    6.364719e-01 


>**Table 5. Thumbs up regression model (significant variables).**

Thumbs up students intercepts model including H1–H11. H1, H2, H4 and H11 significant.

In [24]:
%%R
model <- glmmTMB( thumbs_up ~ H1_first_correct
                            + H2_cumulative_answered
                            + H3_spelling_suggestion
                            + H4_sentence_textrank_rank
                            + H5_answer_tf_idf_rank
                            + H6_answer_pos
                            + H7_answer_log_probability
                            + H8_answer_location
                            + H9_feedback
                            + H10_reviewed
                            + H11_llm_aligned
                            + (1|student_id),
                            family=binomial(link=logit), data=sessions )
summary( model )

 Family: binomial  ( logit )
Formula:          
thumbs_up ~ H1_first_correct + H2_cumulative_answered + H3_spelling_suggestion +  
    H4_sentence_textrank_rank + H5_answer_tf_idf_rank + H6_answer_pos +  
    H7_answer_log_probability + H8_answer_location + H9_feedback +  
    H10_reviewed + H11_llm_aligned + (1 | student_id)
Data: sessions

      AIC       BIC    logLik  deviance  df.resid 
  27739.5   27944.9  -13852.7   27705.5   1305940 

Random effects:

Conditional model:
 Groups     Name        Variance Std.Dev.
 student_id (Intercept) 104.3    10.21   
Number of obs: 1305957, groups:  student_id, 106183

Conditional model:
                            Estimate Std. Error z value Pr(>|z|)    
(Intercept)               -1.311e+01  1.990e-01  -65.89  < 2e-16 ***
H1_first_correct           2.421e-01  5.013e-02    4.83 1.37e-06 ***
H2_cumulative_answered    -1.488e-03  5.139e-04   -2.89   0.0038 ** 
H3_spelling_suggestion    -2.126e-02  9.933e-02   -0.21   0.8305    
H4_sentence_text

Odds ratios.

In [25]:
%%R
exp( fixef( model )$cond )

              (Intercept)          H1_first_correct    H2_cumulative_answered 
             2.025070e-06              1.273916e+00              9.985135e-01 
   H3_spelling_suggestion H4_sentence_textrank_rank     H5_answer_tf_idf_rank 
             9.789662e-01              1.328598e+00              9.491917e-01 
         H6_answer_posADV         H6_answer_posNOUN        H6_answer_posPROPN 
             8.943241e-01              9.286295e-01              1.009971e+00 
        H6_answer_posVERB H7_answer_log_probability        H8_answer_location 
             8.168823e-01              9.844194e-01              1.000338e+00 
       H9_feedbackcontext        H9_feedbackoutcome              H10_reviewed 
             1.112523e+00              1.062310e+00              1.296748e+00 
          H11_llm_aligned 
             1.259234e+00 
